# Filtering bilingual data using character-level edit distance

* Remove near-identical sentence pairs <br>
* Remove heavily paraphrased pairs <br>
* Retain pairs that most likely reflects lexical or morphosyntactic differences <br>

In [42]:
import pandas as pd
import Levenshtein

In [43]:
def edit_ratio(s1, s2):
    # compute character-level edit distance ratio
    dist = Levenshtein.distance(s1, s2)
    # normalize by the length of the longer sentence to get a ratio
    return dist / max(len(s1), len(s2), 1)

# lower ratio indicates more similar sentence pairs
# higher ratio indicates more different sentence pairs

# ratio < 0.25 : too similar (almost identical), REMOVE
# ratio 0.25–0.75 : KEEP
# ratio > 0.75 : too much paraphrase, REMOVE

In [44]:
# filter parallel sentence pairs based on edit distance ratio
def filter_pairs(df, src_col, tgt_col, low=0.25, high=0.75):
    kept = [] # filtered pairs kept for training
    removed = [] # discarded pairs
    
    # iterate through each sentence pair
    for _, row in df.iterrows():
        # convert to string and remove leading/trailing whitespace
        s1 = str(row[src_col]).strip()
        s2 = str(row[tgt_col]).strip()
        
        # compute normalized edit distance ratio
        ratio = edit_ratio(s1, s2)
        
        # keep pairs within the desired range
        if low < ratio < high:
            kept.append((s1, s2, ratio))
        else:
            # remove pairs that are too similar or too different
            removed.append((s1, s2, ratio))
    
    # convert results back to DataFrames
    kept_df = pd.DataFrame(kept, columns=[src_col, tgt_col, "ratio"])
    removed_df = pd.DataFrame(removed, columns=[src_col, tgt_col, "ratio"])

    return kept_df, removed_df

In [45]:
df = pd.read_csv("../data/bilingual/original/train.tsv", sep="\t")

kept_df, removed_df = filter_pairs(df, "nk", "sk")

print("Kept:", len(kept_df))
print("Removed:", len(removed_df))

kept_df.to_csv("../data/bilingual/filtered/train_filtered_ratio.tsv", sep="\t", index=False)

Kept: 84376
Removed: 23117


In [46]:
df = pd.read_csv("../data/bilingual/original/val.tsv", sep="\t")

kept_df, removed_df = filter_pairs(df, "nk", "sk")

print("Kept:", len(kept_df))
print("Removed:", len(removed_df))

kept_df.to_csv("../data/bilingual/filtered/val_filtered_ratio.tsv", sep="\t", index=False)

Kept: 9400
Removed: 2596


In [47]:
df = pd.read_csv("../data/bilingual/original/test.tsv", sep="\t")

kept_df, removed_df = filter_pairs(df, "nk", "sk")

print("Kept:", len(kept_df))
print("Removed:", len(removed_df))

kept_df.to_csv("../data/bilingual/filtered/test_filtered_ratio.tsv", sep="\t", index=False)

Kept: 8703
Removed: 2546


create .tsv files without the 'ratio' columns (later used to train models)

In [48]:
def remove_ratio(input_path, output_path):
    df = pd.read_csv(input_path, sep="\t")
    
    # drop ratio column if it exists
    if "ratio" in df.columns:
        df = df.drop(columns=["ratio"])
    
    df.to_csv(output_path, sep="\t", index=False)

# apply to all splits
remove_ratio("../data/bilingual/filtered/train_filtered_ratio.tsv",
             "../data/bilingual/filtered/train_filtered.tsv")

remove_ratio("../data/bilingual/filtered/val_filtered_ratio.tsv",
             "../data/bilingual/filtered/val_filtered.tsv")

remove_ratio("../data/bilingual/filtered/test_filtered_ratio.tsv",
             "../data/bilingual/filtered/test_filtered.tsv")